# 🔥 PyTorch 기초 실습

## 1. 환경 설정 & PyTorch 소개

PyTorch는 딥러닝 연구를 위한 오픈소스 프레임워크로, 마치 '연구자를 위한 수치 계산 도구'라고 생각하시면 됩니다. NumPy처럼 쉽게 쓸 수 있으면서, GPU 가속과 자동 미분 기능을 제공합니다.

먼저 필요한 라이브러리를 불러오고 환경을 설정합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# 한국어 폰트 설정 (Colab 환경)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# PyTorch 버전 확인
print(f"PyTorch 버전: {torch.__version__}")

# 재현성을 위한 시드 설정
torch.manual_seed(42)
np.random.seed(42)
print("시드 설정 완료 ✓")

# GPU 사용 가능 여부 확인
if torch.cuda.is_available():
    print(f"GPU 사용 가능: {torch.cuda.get_device_name(0)}")
else:
    print("GPU 사용 불가 — CPU로 실행합니다.")

---
## 2. 텐서(Tensor) 기초

텐서는 **GPU 버전의 NumPy 배열**이라고 생각하시면 됩니다. 스칼라(0차원), 벡터(1차원), 행렬(2차원), 그 이상의 다차원 배열 모두 텐서로 표현합니다.

### 주요 텐서 생성 방법

| 함수 | 설명 |
|------|------|
| `torch.zeros(m, n)` | 0으로 채워진 m×n 텐서 |
| `torch.ones(m, n)` | 1로 채워진 m×n 텐서 |
| `torch.rand(m, n)` | [0,1) 균등분포 난수 |
| `torch.randn(m, n)` | 표준정규분포 난수 |
| `torch.arange(start, end, step)` | 등간격 1D 텐서 |
| `torch.tensor(list)` | 파이썬 리스트/NumPy 배열에서 생성 |
| `torch.from_numpy(arr)` | NumPy 배열과 메모리 공유 |

In [ ]:
# 텐서 생성 예시
zeros = torch.zeros(3, 4)       # 0으로 채워진 3x4 텐서
ones = torch.ones(2, 3)         # 1로 채워진 2x3 텐서
rand_t = torch.rand(2, 2)       # [0, 1) 균등분포 난수
randn_t = torch.randn(3, 3)     # 표준정규분포 난수

arange_t = torch.arange(0, 10, 2)  # 0부터 10미만까지 2 간격

# 파이썬 리스트에서 생성
list_t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

# NumPy 배열에서 생성
np_arr = np.array([1.0, 2.0, 3.0])
from_np = torch.from_numpy(np_arr)

print("zeros:\n", zeros)
print("\nones:\n", ones)
print("\narange:", arange_t)
print("\nlist_t:\n", list_t)
print("\nfrom_np:", from_np)

### ✏️ 실습 2-1: 텐서 생성

In [ ]:
# TODO: 아래 빈칸을 채워서 요구된 텐서를 만들어 보세요.

# (1) 5x5 크기의 0 텐서 생성
my_zeros = _______________

# (2) (3, 4) 형태의 1 텐서 생성
my_ones = _______________

# (3) [10, 20, 30, 40, 50] 값을 가진 1D 텐서 생성
my_tensor = _______________

# 검증
assert my_zeros.shape == (5, 5), "my_zeros의 shape이 (5, 5)여야 합니다!"
assert my_ones.shape == (3, 4), "my_ones의 shape이 (3, 4)여야 합니다!"
assert my_tensor.tolist() == [10, 20, 30, 40, 50], "my_tensor 값을 확인하세요!"
print("✅ 모두 정답입니다!")

### Shape & dtype 확인

In [ ]:
# shape 확인
t = torch.randn(2, 3, 4)
print("shape:", t.shape)
print("size():", t.size())
print("차원 수:", t.ndim)
print("전체 원소 수:", t.numel())

In [ ]:
# dtype 확인 및 변환
int_t = torch.tensor([1, 2, 3])
print("기본 dtype:", int_t.dtype)

float_t = int_t.float()          # float32로 변환
long_t = float_t.long()          # int64로 변환
float32_t = int_t.to(torch.float32)

print("float():", float_t.dtype)
print("long():", long_t.dtype)
print("to(torch.float32):", float32_t.dtype)

---
## 3. 텐서 연산

텐서 연산은 NumPy와 매우 유사합니다. 원소별(element-wise) 연산, 행렬 곱셈, shape 변환 등을 다룹니다.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("덧셈:", a + b)
print("뺄셈:", a - b)
print("element-wise 곱셈:", a * b)
print("나눗셈:", a / b)

### In-place 연산

`_`로 끝나는 연산은 **in-place 연산**으로, 새 텐서를 만들지 않고 현재 텐서를 직접 수정합니다.

In [ ]:
# in-place 연산 (메모리를 새로 할당하지 않고 직접 수정)
c = torch.tensor([1.0, 2.0, 3.0])
print("변경 전:", c)
c.add_(10)  # c = c + 10
print("add_(10) 후:", c)
# ⚠️ 주의: in-place 연산은 autograd와 함께 사용할 때 문제가 생길 수 있습니다.

### Shape 변환

딥러닝에서 가장 자주 하는 작업 중 하나입니다. 데이터를 모델에 넣기 위해 shape을 맞춰야 합니다.

In [ ]:
t = torch.arange(24).float()  # 0~23까지 24개 원소

# view: 원소 총 개수가 같아야 함 (메모리 연속적)
t_2d = t.view(4, 6)
print("view(4, 6):\n", t_2d)

# reshape: view와 유사하지만 메모리 비연속 텐서도 처리 가능
t_3d = t.reshape(2, 3, 4)
print("\nreshape(2, 3, 4) shape:", t_3d.shape)

# permute: 차원 순서 변경 (예: 이미지 CHW -> HWC)
img = torch.randn(3, 32, 32)  # 채널, 높이, 너비
img_hwc = img.permute(1, 2, 0)  # 높이, 너비, 채널
print("\npermute(1,2,0):", img.shape, "->", img_hwc.shape)

# unsqueeze: 차원 추가
t1d = torch.tensor([1.0, 2.0, 3.0])
t2d = t1d.unsqueeze(0)  # 앞에 차원 추가 -> (1, 3)
print("\nunsqueeze(0):", t1d.shape, "->", t2d.shape)

# squeeze: 크기가 1인 차원 제거
t_sq = t2d.squeeze(0)
print("squeeze(0):", t2d.shape, "->", t_sq.shape)

### ✏️ 실습 3-1: Shape 변환

In [ ]:
# TODO: 아래 연산들을 완성하세요.

t = torch.arange(12).float()

# (1) t를 (3, 4) 형태로 변환
t_3x4 = _______________

# (2) t_3x4를 (2, 6) 형태로 변환
t_2x6 = _______________

# (3) t를 (2, 2, 3) 형태로 변환
t_2x2x3 = _______________

# (4) shape가 (3, 4)인 텐서에 앞쪽에 차원을 추가해서 (1, 3, 4) 만들기
t_1x3x4 = _______________

assert t_3x4.shape == (3, 4), f"shape 오류: {t_3x4.shape}"
assert t_2x6.shape == (2, 6), f"shape 오류: {t_2x6.shape}"
assert t_2x2x3.shape == (2, 2, 3), f"shape 오류: {t_2x2x3.shape}"
assert t_1x3x4.shape == (1, 3, 4), f"shape 오류: {t_1x3x4.shape}"
print("✅ shape 변환 모두 정답!")

### 행렬 곱셈 (Matrix Multiplication)

딥러닝 연산의 핵심! `@` 연산자를 사용하면 직관적으로 표현할 수 있습니다.

In [ ]:
# 행렬 곱셈
A = torch.randn(3, 4)
B = torch.randn(4, 5)

C1 = torch.matmul(A, B)     # 방법 1: torch.matmul
C2 = A @ B                   # 방법 2: @ 연산자 (추천!)

print("A.shape:", A.shape)
print("B.shape:", B.shape)
print("A @ B shape:", C2.shape)
print("두 결과 동일:", torch.allclose(C1, C2))

### ✏️ 실습 3-2: 배치 행렬 곱셈

In [ ]:
# TODO: 행렬 곱셈을 완성하세요.

# 배치 행렬 곱셈: (2, 3, 4) @ (2, 4, 5) -> (2, 3, 5)
X = torch.randn(2, 3, 4)
Y = torch.randn(2, 4, 5)

Z = _______________  # X와 Y의 배치 행렬 곱셈

assert Z.shape == (2, 3, 5), f"shape 오류: {Z.shape}"
print("✅ 배치 행렬 곱셈 정답!")

### ✏️ 실습 3-3: 인덱싱 & 슬라이싱

NumPy와 동일한 문법을 사용합니다.

In [ ]:
t = torch.arange(20).reshape(4, 5).float()
print("원본 텐서:\n", t)

# TODO: 아래 인덱싱/슬라이싱을 완성하세요.

# (1) 2번째 행(index=1) 전체 추출
row1 = _______________

# (2) 마지막 열 전체 추출
last_col = _______________

# (3) 왼쪽 위 2x3 부분 추출
top_left = _______________

# (4) 짝수 행만 추출 (0, 2번째 행)
even_rows = _______________

assert row1.tolist() == [5.0, 6.0, 7.0, 8.0, 9.0], "row1 오류"
assert last_col.tolist() == [4.0, 9.0, 14.0, 19.0], "last_col 오류"
assert top_left.shape == (2, 3), f"top_left shape 오류: {top_left.shape}"
assert even_rows.shape == (2, 5), f"even_rows shape 오류: {even_rows.shape}"
print("✅ 인덱싱 모두 정답!")

---
## 4. Autograd: 자동 미분

딥러닝 학습의 핵심은 **손실 함수를 파라미터에 대해 미분**하는 것입니다. PyTorch는 이를 자동으로 계산해 줍니다.

마치 연구노트에 실험 과정을 기록하듯, PyTorch는 **계산 그래프(Computational Graph)**에 모든 연산을 기록합니다. 그 후 `backward()`를 호출하면 기록된 경로를 역방향으로 거슬러 올라가며 gradient를 계산합니다.

- `requires_grad=True`: "이 텐서의 gradient를 추적해줘"
- `.backward()`: gradient 계산 (역전파)
- `.grad`: 계산된 gradient 저장소

In [ ]:
# requires_grad=True: "이 텐서에 대해 미분을 추적해줘"
x = torch.tensor([2.0, 4.0], requires_grad=True)

# y = (x + 2)^2 의 평균
y = ((x + 2) ** 2).mean()
print("x:", x)
print("y:", y)

# 역전파: y를 x에 대해 미분
y.backward()

# gradient 확인: dy/dx = 2*(x+2)/2 = x+2
# x=[2,4]이면 grad=[4, 6]
print("x.grad:", x.grad)
print("손계산 확인: x+2 =", (x + 2).detach())

### ✏️ 실습 4-1: Gradient 계산

In [ ]:
# TODO: 아래 함수의 gradient를 계산해 보세요.
# f(x) = 3*x^2 + 2*x + 1 의 x=3에서의 미분값을 구하세요.
# 손계산: f'(x) = 6x + 2, x=3에서 f'(3) = 20

x = torch.tensor([3.0], requires_grad=True)

# f(x) 계산
f = _______________

# 역전파
_______________

# gradient 출력
print("x.grad:", x.grad)

assert abs(x.grad.item() - 20.0) < 1e-4, f"gradient 오류: {x.grad.item()} (정답: 20.0)"
print("✅ Autograd 정답!")

### torch.no_grad()

평가/추론 시에는 gradient 계산이 필요 없습니다. `torch.no_grad()`를 사용하면 메모리와 연산을 절약할 수 있습니다.

In [ ]:
# torch.no_grad(): 평가/추론 시 gradient 계산 비활성화
# -> 메모리 절약, 속도 향상
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

with torch.no_grad():
    y = x * 2  # gradient 추적 없음
    print("no_grad 내부 y.requires_grad:", y.requires_grad)

print("\n🤔 생각해보기: 왜 모델 평가 시에는 torch.no_grad()가 필요할까요?")
print("   힌트: 학습 중에는 gradient가 필요하지만, 예측만 할 때는...")

---
## 5. GPU 활용

**CPU vs GPU 비유:**
- CPU: 똑똑한 교수님 10명 (복잡한 순차 작업에 강함)
- GPU: 열심히 일하는 학생 10,000명 (단순 작업을 동시에 처리하는 병렬 처리 장치)

딥러닝의 핵심인 행렬 연산은 독립적인 계산이 많아 GPU에 매우 적합합니다.

In [ ]:
# device 설정 패턴 (표준 관용구)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

### ✏️ 실습 5-1: 텐서를 GPU로 이동

In [ ]:
# TODO: 텐서를 device로 옮기세요.

t = torch.randn(3, 3)
print("원래 device:", t.device)

# t를 device로 이동
t_on_device = _______________

print("이동 후 device:", t_on_device.device)
assert str(t_on_device.device).startswith(str(device).split(':')[0]), "device 이동 오류"
print("✅ device 이동 성공!")

### CPU vs GPU 속도 비교

In [ ]:
import time

size = 1000

# CPU 행렬 곱
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
for _ in range(10):
    _ = a_cpu @ b_cpu
cpu_time = time.time() - start
print(f"CPU 시간: {cpu_time:.4f}초")

# GPU 행렬 곱 (GPU 사용 가능한 경우)
if torch.cuda.is_available():
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)
    
    # GPU 워밍업
    _ = a_gpu @ b_gpu
    torch.cuda.synchronize()
    
    start = time.time()
    for _ in range(10):
        _ = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU 시간: {gpu_time:.4f}초")
    print(f"GPU가 CPU보다 {cpu_time/gpu_time:.1f}배 빠릅니다!")
else:
    print("GPU 없음 — CPU만 사용합니다.")

### ⚠️ GPU 텐서 → NumPy 변환 주의사항

GPU 텐서를 바로 NumPy로 변환하면 에러가 발생합니다. 올바른 변환 방법을 익혀두세요.

In [ ]:
print("⚠️ GPU 텐서를 NumPy로 변환하는 방법")
print()

t_gpu = torch.randn(3, 3).to(device)

# 잘못된 방법 (GPU 텐서에서 바로 numpy() 호출)
print("❌ 잘못된 방법:")
try:
    wrong = t_gpu.numpy()  # CPU 텐서에서만 가능
except RuntimeError as e:
    print(f"   에러 발생: {e}")

print()
# 올바른 방법
print("✅ 올바른 방법:")
correct = t_gpu.cpu().detach().numpy()
print(f"   numpy 변환 성공! shape: {correct.shape}")

### ✏️ 실습 5-2: GPU 텐서 → NumPy 변환

In [ ]:
# TODO: GPU 텐서를 NumPy 배열로 올바르게 변환하세요.

t = torch.randn(4, 4, requires_grad=True).to(device)

# t를 NumPy 배열로 변환 (requires_grad와 GPU 모두 처리)
t_numpy = _______________

assert isinstance(t_numpy, np.ndarray), "NumPy 배열이 아닙니다!"
assert t_numpy.shape == (4, 4), "shape 오류"
print("✅ GPU 텐서 → NumPy 변환 성공!")

---
## 6. 신경망 만들기 — nn.Module

`nn.Module`은 **레고 블록처럼 레이어를 쌓아 신경망을 만드는 클래스**입니다.

반드시 구현해야 하는 두 가지:
1. `__init__`: 사용할 레이어(파라미터)를 정의
2. `forward`: 데이터가 레이어를 통과하는 순서(흐름)를 정의

### 자주 사용하는 레이어

| 레이어 | 설명 |
|--------|------|
| `nn.Linear(in, out)` | 완전 연결층 (y = Wx + b) |
| `nn.ReLU()` | 활성화 함수: max(0, x) |
| `nn.Tanh()` | 활성화 함수: tanh(x) |
| `nn.Sigmoid()` | 활성화 함수: σ(x) |
| `nn.BatchNorm1d(dim)` | 배치 정규화 |
| `nn.Dropout(p)` | 드롭아웃 (과적합 방지) |

In [ ]:
# 간단한 신경망 예시
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # __init__에서 레이어 정의
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.linear2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # forward에서 데이터 흐름 정의
        x = self.linear1(x)
        x = self.act(x)
        x = self.linear2(x)
        return x

model = SimpleNet(input_dim=4, hidden_dim=8, output_dim=1)
print(model)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
print(f"\n전체 파라미터 수: {total_params:,}")

### ✏️ 실습 6-1: SimpleClassifier 구현

In [ ]:
# TODO: SimpleClassifier 클래스의 빈칸을 채우세요.
# 구조: Linear(2→16) -> Tanh -> Linear(16→1)

class SimpleClassifier(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # TODO: (1) 첫 번째 선형 레이어 정의 (num_inputs -> num_hidden)
        self.linear1 = _______________
        # TODO: (2) 활성화 함수 (Tanh 사용)
        self.act_fn = _______________
        # TODO: (3) 두 번째 선형 레이어 정의 (num_hidden -> num_outputs)
        self.linear2 = _______________
    
    def forward(self, x):
        # TODO: (4) 순전파 구현 (linear1 -> act_fn -> linear2)
        x = _______________
        x = _______________
        x = _______________
        return x

# 검증
model = SimpleClassifier(num_inputs=2, num_hidden=16, num_outputs=1)
test_input = torch.randn(5, 2)
test_output = model(test_input)
assert test_output.shape == (5, 1), f"출력 shape 오류: {test_output.shape}"
print("모델 구조:")
print(model)
print(f"\n✅ SimpleClassifier 구현 완료! (입력: {test_input.shape} → 출력: {test_output.shape})")

---
## 7. 데이터 준비 — Dataset & DataLoader

- **Dataset**: 데이터를 PyTorch가 이해하는 형태로 포장하는 클래스
  - `__len__`: 전체 데이터 수 반환
  - `__getitem__`: 인덱스로 개별 샘플 반환

- **DataLoader**: Dataset을 받아 배치(batch) 단위로 공급하는 이터레이터
  - 자동 배치 생성, 셔플, 병렬 로딩 지원

### XOR 데이터셋

XOR(배타적 논리합): 두 입력이 **다르면** 1, **같으면** 0을 출력합니다.

| x1 | x2 | XOR |
|----|----|-----|
| 0  | 0  | 0   |
| 1  | 0  | 1   |
| 0  | 1  | 1   |
| 1  | 1  | 0   |

단순 선형 분류기로는 풀 수 없는 문제입니다. 🤔

In [ ]:
# XOR 데이터셋 (연속 버전)
# XOR: 두 입력이 다르면 1, 같으면 0 — 선형 분류기로는 풀 수 없는 문제!

class XORDataset(Dataset):
    def __init__(self, size, std=0.1):
        """
        size: 데이터 샘플 수
        std: 노이즈의 표준편차
        """
        super().__init__()
        self.size = size
        # XOR의 4가지 기준점: (0,0)→0, (1,0)→1, (0,1)→1, (1,1)→0
        np.random.seed(42)
        self.data = np.zeros((size, 2), dtype=np.float32)
        self.label = np.zeros(size, dtype=np.float32)
        
        for i in range(size):
            x1 = np.random.randint(0, 2)
            x2 = np.random.randint(0, 2)
            # 노이즈 추가
            self.data[i] = [x1 + np.random.normal(0, std),
                            x2 + np.random.normal(0, std)]
            self.label[i] = float(x1 ^ x2)  # XOR
    
    def __len__(self):
        # TODO: 데이터셋 크기 반환
        return _______________
    
    def __getitem__(self, idx):
        # TODO: idx번째 (데이터, 레이블) 반환 (torch.Tensor로 변환)
        data_point = _______________
        label = _______________
        return data_point, label

# 검증
dataset = XORDataset(size=200)
assert len(dataset) == 200, "len 오류"
sample_data, sample_label = dataset[0]
assert isinstance(sample_data, torch.Tensor), "data가 Tensor가 아닙니다"
assert isinstance(sample_label, torch.Tensor), "label이 Tensor가 아닙니다"
assert sample_data.shape == (2,), f"data shape 오류: {sample_data.shape}"
print(f"✅ Dataset 구현 완료! 첫 번째 샘플: data={sample_data}, label={sample_label}")

In [ ]:
# DataLoader: 배치 생성, 셔플, 병렬 로딩
train_dataset = XORDataset(size=2000)
val_dataset = XORDataset(size=500)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print(f"학습 배치 수: {len(train_loader)}")
print(f"검증 배치 수: {len(val_loader)}")

# 첫 번째 배치 확인
first_batch = next(iter(train_loader))
print(f"\n첫 배치 데이터 shape: {first_batch[0].shape}")
print(f"첫 배치 레이블 shape: {first_batch[1].shape}")

In [ ]:
# 데이터 시각화
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
data_np = train_dataset.data
labels_np = train_dataset.label

# 클래스별 색상 구분
for cls, color, label_name in [(0, 'blue', 'Class 0'), (1, 'red', 'Class 1')]:
    mask = labels_np == cls
    ax.scatter(data_np[mask, 0], data_np[mask, 1], 
               c=color, alpha=0.5, s=20, label=label_name)

ax.set_xlabel('X1')
ax.set_ylabel('X2')
ax.set_title('XOR 데이터셋 시각화')
ax.legend()
plt.tight_layout()
plt.show()
print("🤔 생각해보기: 직선 하나로 빨간 점과 파란 점을 분리할 수 있을까요?")

---
## 8. 학습 루프

모든 딥러닝 학습은 다음 5단계를 반복합니다:

| 단계 | 설명 |
|------|------|
| 1️⃣ 데이터 로드 | 배치 가져오기 & device 이동 |
| 2️⃣ 예측 (순전파) | 모델에 데이터를 통과시킴 |
| 3️⃣ 손실 계산 | 예측값과 정답의 차이 계산 |
| 4️⃣ 역전파 | gradient 계산 |
| 5️⃣ 파라미터 업데이트 | optimizer가 가중치 갱신 |

> 💡 **중요**: `optimizer.zero_grad()`를 매 배치마다 호출해야 합니다. 그렇지 않으면 gradient가 누적되어 잘못된 업데이트가 발생합니다.

In [ ]:
# 모델, 손실 함수, 옵티마이저 설정
model = SimpleClassifier(num_inputs=2, num_hidden=16, num_outputs=1)
model = model.to(device)

# BCEWithLogitsLoss = Sigmoid + Binary Cross Entropy
# 수치적으로 더 안정적 (logit을 직접 받음)
loss_module = nn.BCEWithLogitsLoss()

# SGD 옵티마이저
optimizer = optim.SGD(model.parameters(), lr=0.1)

print("모델 설정 완료!")
print(f"장치: {device}")

### ✏️ 실습 8-1: 학습 루프 구현

In [ ]:
def train_model(model, optimizer, data_loader, loss_module, num_epochs=100):
    model.train()  # 학습 모드
    loss_history = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for data_inputs, data_labels in data_loader:
            # TODO: 1단계 — 데이터를 device로 이동
            data_inputs = _______________
            data_labels = _______________
            
            # TODO: 2단계 — 모델 예측 (순전파)
            preds = _______________
            preds = preds.squeeze(dim=1)  # (batch, 1) -> (batch,)
            
            # TODO: 3단계 — 손실 계산
            loss = _______________
            
            # TODO: 4단계 — gradient 초기화 및 역전파
            _______________  # gradient 초기화
            _______________  # 역전파
            
            # TODO: 5단계 — 파라미터 업데이트
            _______________
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(data_loader)
        loss_history.append(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs} | 손실: {avg_loss:.4f}")
    
    return loss_history

# 학습 실행
print("학습 시작...")
loss_history = train_model(model, optimizer, train_loader, loss_module, num_epochs=100)
print("학습 완료!")

In [ ]:
# 손실 그래프
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('학습 손실 변화')
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 9. 평가 & 시각화

학습이 끝난 모델을 평가합니다. 평가 시에는 반드시:
1. `model.eval()` — 학습 전용 레이어(BatchNorm, Dropout) 비활성화
2. `@torch.no_grad()` 또는 `with torch.no_grad()` — gradient 계산 비활성화

In [ ]:
@torch.no_grad()  # 평가 시 gradient 계산 불필요
def eval_model(model, data_loader):
    model.eval()  # 평가 모드 (BatchNorm, Dropout 등에 영향)
    
    correct = 0
    total = 0
    
    for data_inputs, data_labels in data_loader:
        # TODO: (1) 데이터를 device로 이동
        data_inputs = _______________
        data_labels = _______________
        
        # TODO: (2) 예측
        preds = _______________
        preds = preds.squeeze(dim=1)
        
        # TODO: (3) sigmoid 적용 후 0.5 기준으로 이진 분류
        pred_labels = _______________  # 힌트: torch.sigmoid(preds) > 0.5
        
        # TODO: (4) 정확도 계산
        correct += _______________  # 정답 수
        total += data_labels.shape[0]
    
    accuracy = correct / total
    return accuracy

train_acc = eval_model(model, train_loader)
val_acc = eval_model(model, val_loader)
print(f"학습 정확도: {train_acc:.4f} ({train_acc*100:.1f}%)")
print(f"검증 정확도: {val_acc:.4f} ({val_acc*100:.1f}%)")

In [ ]:
# 결정 경계 시각화
@torch.no_grad()
def plot_decision_boundary(model, dataset, device):
    model.eval()
    
    # meshgrid 생성
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    
    grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).to(device)
    probs = torch.sigmoid(model(grid)).cpu().numpy().reshape(xx.shape)
    
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.contourf(xx, yy, probs, alpha=0.3, cmap='RdBu_r', levels=20)
    ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
    
    data_np = dataset.data
    labels_np = dataset.label
    for cls, color, name in [(0, 'blue', 'Class 0'), (1, 'red', 'Class 1')]:
        mask = labels_np == cls
        ax.scatter(data_np[mask, 0], data_np[mask, 1],
                   c=color, alpha=0.6, s=15, label=name)
    
    ax.set_title('결정 경계 시각화')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_decision_boundary(model, val_dataset, device)

### 모델 저장 & 불러오기

`state_dict()`는 모델의 파라미터(가중치, 편향)를 담은 딕셔너리입니다.

In [ ]:
import os

# 모델 저장
save_path = "/tmp/simple_classifier.pth"
torch.save(model.state_dict(), save_path)
print(f"✅ 모델 저장 완료: {save_path}")

# 모델 불러오기
loaded_model = SimpleClassifier(num_inputs=2, num_hidden=16, num_outputs=1)
loaded_model.load_state_dict(torch.load(save_path, map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()
print("✅ 모델 불러오기 완료!")

# 동일한 결과 확인
test_input = torch.randn(10, 2).to(device)
with torch.no_grad():
    out1 = model(test_input)
    out2 = loaded_model(test_input)
print(f"원본 = 불러온 모델: {torch.allclose(out1, out2)}")

---
## 10. 🔥 도전 과제: Image Pixel Surgery

지금까지 배운 모든 내용을 종합 적용합니다!

- 텐서 생성 → PIL 이미지 → torchvision transforms → GPU 전송 → GPU 텐서 조작 → NumPy 역변환 → 시각화

외부 이미지 파일 없이 코드로 직접 이미지를 생성합니다.

In [ ]:
from torchvision import transforms
from PIL import Image

# --- 1단계: 256x256 그라데이션 이미지 생성 ---
# 외부 파일 불필요! 코드로 직접 생성
height, width = 256, 256

# RGB 채널별 그라데이션 생성
r_channel = np.linspace(0, 255, width, dtype=np.uint8)[np.newaxis, :].repeat(height, axis=0)
g_channel = np.linspace(0, 255, height, dtype=np.uint8)[:, np.newaxis].repeat(width, axis=1)
b_channel = np.full((height, width), 100, dtype=np.uint8)

img_np = np.stack([r_channel, g_channel, b_channel], axis=2)
original_image = Image.fromarray(img_np, mode='RGB')

print(f"원본 이미지 크기: {original_image.size}")

# 시각화
plt.figure(figsize=(4, 4))
plt.imshow(original_image)
plt.title("원본 이미지 (그라데이션)")
plt.axis('off')
plt.show()

In [ ]:
# --- 2단계: PIL Image → Tensor 변환 ---
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),                   # [0,255] → [0,1], HWC → CHW
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])  # [-1, 1]로 정규화
])

img_tensor = transform(original_image)
print(f"변환 후 텐서 shape: {img_tensor.shape}")  # (3, 256, 256)
print(f"값 범위: [{img_tensor.min():.2f}, {img_tensor.max():.2f}]")

assert img_tensor.shape == (3, 256, 256), f"shape 오류: {img_tensor.shape}"
print("✅ PIL → Tensor 변환 완료!")

In [ ]:
# --- 3단계: CPU → GPU 전송 ---
img_tensor = img_tensor.to(device)
print(f"텐서 장치: {img_tensor.device}")
assert str(img_tensor.device).startswith(str(device).split(':')[0])
print("✅ GPU 전송 완료!")

In [ ]:
# --- 4단계: GPU에서 이미지 조작 ---

# (a) 채널 스왑: RGB → BGR
img_bgr = img_tensor[[2, 1, 0], :, :]  # 채널 인덱싱으로 순서 변경
print(f"채널 스왑 후 shape: {img_bgr.shape}")

# (b) 좌우 반전
img_flipped = img_tensor.flip(dims=[2])  # 너비 축(dim=2)을 기준으로 반전
print(f"좌우 반전 후 shape: {img_flipped.shape}")

# (c) 밝기 조절 (+0.3) + clamp로 범위 고정
img_bright = (img_tensor + 0.3).clamp(-1.0, 1.0)
print(f"밝기 조절 후 범위: [{img_bright.min():.2f}, {img_bright.max():.2f}]")

assert img_bgr.shape == img_tensor.shape
assert img_flipped.shape == img_tensor.shape
assert img_bright.shape == img_tensor.shape
print("✅ 이미지 조작 완료!")

In [ ]:
# --- 5단계: GPU Tensor → NumPy 역변환 ---

def tensor_to_numpy_image(tensor):
    """GPU Tensor (C, H, W), 정규화됨 -> NumPy (H, W, C), [0, 255]"""
    # 역정규화: [-1,1] -> [0,1]
    tensor = tensor * 0.5 + 0.5
    # CHW -> HWC
    tensor = tensor.permute(1, 2, 0)
    # GPU -> CPU -> NumPy
    img_np = tensor.cpu().detach().numpy()
    # [0,1] -> [0,255]
    img_np = (img_np * 255).clip(0, 255).astype(np.uint8)
    return img_np

original_np = tensor_to_numpy_image(img_tensor)
bgr_np = tensor_to_numpy_image(img_bgr)
flipped_np = tensor_to_numpy_image(img_flipped)
bright_np = tensor_to_numpy_image(img_bright)

print(f"역변환 후 shape: {original_np.shape}, dtype: {original_np.dtype}")
assert original_np.shape == (256, 256, 3)
assert original_np.dtype == np.uint8
print("✅ GPU Tensor → NumPy 역변환 완료!")

In [ ]:
# --- 6단계: 결과 시각화 ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['원본', 'BGR 채널 스왑', '좌우 반전', '밝기 증가']
images = [original_np, bgr_np, flipped_np, bright_np]

for ax, title, img in zip(axes, titles, images):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Image Pixel Surgery 결과', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 보너스: 픽셀 차이 히트맵
diff = np.abs(original_np.astype(np.int32) - bright_np.astype(np.int32)).mean(axis=2)

plt.figure(figsize=(6, 5))
im = plt.imshow(diff, cmap='hot')
plt.colorbar(im, label='평균 픽셀 차이')
plt.title('원본 vs 밝기 증가 — 픽셀 차이 히트맵')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"평균 픽셀 차이: {diff.mean():.2f}")
print(f"최대 픽셀 차이: {diff.max():.2f}")

In [ ]:
print("=" * 50)
print("🎉 PyTorch 기초 실습 완료!")
print("=" * 50)
print("""
오늘 배운 내용:
  ✅ 텐서 생성 및 조작
  ✅ Autograd (자동 미분)
  ✅ GPU 활용
  ✅ nn.Module로 신경망 구현
  ✅ Dataset & DataLoader
  ✅ 학습 루프 (5단계)
  ✅ 모델 평가 및 시각화
  ✅ Image Pixel Surgery (종합 실습)
""")